[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/warlockee/oxRL/blob/main/examples/oxrl_quickstart.ipynb)

# oxRL Quickstart: Post-train any LLM in 10 lines of code

This notebook demonstrates how to post-train a language model using **oxRL** -- a lightweight framework with 51 algorithms for RL and preference optimization.

**What you will do:**
1. Train Qwen2.5-0.5B-Instruct on math with GRPO (RL)
2. Run DPO preference optimization (SL)
3. See the full algorithm catalog

**Requirements:** A single GPU (T4 or better). Runs in under 5 minutes on Colab.

---
## 1. Install oxRL

In [ ]:
!pip install oxrl -q

---
## 2. RL Training with GRPO

GRPO (Group Relative Policy Optimization) trains a policy using online rollouts scored by a reward function. No value head or reward model needed -- just a model, a task, and a reward signal.

Here we train **Qwen2.5-0.5B-Instruct** on the GSM8K math task. The `Trainer` API auto-configures everything: DeepSpeed ZeRO-3, rollout generation via vLLM, reward scoring, and the training loop.

In [ ]:
%%time
from oxrl import Trainer

trainer = Trainer(model="Qwen/Qwen2.5-0.5B-Instruct")
trainer.train(task="math", steps_per_epoch=5)

That is it. Under the hood, `Trainer` did the following:
1. Auto-generated a YAML config (model size, batch sizes, DeepSpeed settings)
2. Launched `main_rl.py` with Ray for distributed rollout + training
3. Used `gsm8k_reward_func` to score rollouts with binary numeric matching
4. Ran 5 optimizer steps of SGRPO (token-level clipped surrogate)

You can change the task to `"code"`, `"reasoning"`, `"instruct"`, `"vision"`, or `"audio"`.

---
## 3. DPO Preference Optimization (SL path)

oxRL also supports 45 supervised/offline preference algorithms through `main_sl.py`. These train on static preference datasets (chosen vs. rejected pairs) without rollouts.

Below is a minimal DPO config. In production, you would launch this with:
```bash
deepspeed --num_gpus 1 main_sl.py --config-file config.yaml
```

In [ ]:
%%writefile /tmp/dpo_config.yaml
# Minimal DPO config for Qwen2.5-0.5B-Instruct
run:
  experiment_id: quickstart_dpo
  distributed_training_strategy: deepspeed-zero3
  checkpoint_dir: ./checkpoints/quickstart_dpo
  project_name: oxrl-exp
  tracking_uri: ""
  seed: 42

train:
  alg_name: dpo
  lr: 1.0e-05
  total_number_of_epochs: 1
  micro_batches_per_epoch: 20
  train_batch_size_per_gpu: 2
  gradient_accumulation_steps: 4
  val_batch_size_per_gpu: 16
  beta: 0.1

model:
  name: Qwen/Qwen2.5-0.5B-Instruct
  dtype: bfloat16
  ref_model: Qwen/Qwen2.5-0.5B-Instruct
  trust_remote_code: true
  use_cache: false
  model_class: llm
  gradient_checkpointing: true
  attn_implementation: eager

data:
  train_dnames: [preference_train]
  train_files_path: ./data/preference_train.parquet
  val_files_path: ./data/preference_val.parquet
  num_workers: 4
  max_seq_len: 512
  prompt_key: prompt
  answer_key: answer

deepspeed:
  zero_optimization:
    stage: 3
    stage3_param_persistence_threshold: 100000.0
    stage3_prefetch_bucket_size: 50000000.0
    offload_optimizer:
      device: none
      pin_memory: true
    offload_param:
      device: none
      pin_memory: true
    contiguous_gradients: true
    overlap_comm: true
    reduce_scatter: true
    reduce_bucket_size: 500000000.0
    allgather_bucket_size: 500000000.0
    stage3_gather_16bit_weights_on_model_save: true
  activation_checkpointing:
    partition_activations: true
    contiguous_memory_optimization: true
  steps_per_print: 100
  wall_clock_breakdown: false

In [ ]:
%%time
# Launch DPO training via DeepSpeed
# (requires a preference dataset at the paths above)
!deepspeed --num_gpus 1 main_sl.py --config-file /tmp/dpo_config.yaml --experiment_id quickstart_dpo

To switch algorithms, just change `alg_name` in the config. For example:
- `alg_name: simpo` -- reference-free, no ref model needed
- `alg_name: orpo` -- odds ratio preference, reference-free
- `alg_name: kto` -- Kahneman-Tversky optimization
- `alg_name: ipo` -- identity preference optimization

All 45 SL algorithms use the same config structure. See `registry/examples/` for ready-made configs.

---
## 4. Algorithm Catalog (51 algorithms)

oxRL ships with **51 algorithms** across two training paradigms.

### RL Algorithms (online, policy-gradient via `main_rl.py`)

These generate rollouts online using vLLM, score them with a reward function, and update the policy.

| # | Algorithm | Key | Description |
|---|-----------|-----|-------------|
| 1 | **SGRPO** | `sgrpo` | Token-level clipped surrogate (default for dense models) |
| 2 | **GSPO** | `gspo` | Sequence-level clipped surrogate (for MoE models) |
| 3 | **CISPO** | `cispo` | Clipped ratio weighted log-prob (conservative variant) |
| 4 | **RLHF** | `rlhf` | RL from Human Feedback (alias for SGRPO + reward model) |
| 5 | **RLAIF** | `rlaif` | RL from AI Feedback (alias for SGRPO) |
| 6 | **PPO** | `ppo` | Proximal Policy Optimization with value head |

### SL Algorithms (offline, preference/supervised via `main_sl.py`)

These train on static datasets without rollout generation.

#### Core Training

| # | Algorithm | Key | Description |
|---|-----------|-----|-------------|
| 7 | **SFT** | `sft` | Supervised fine-tuning |
| 8 | **CPT** | `cpt` | Continued pre-training |
| 9 | **KD** | `kd` | Knowledge distillation |
| 10 | **RFT** | `rft` | Rejection sampling fine-tuning |
| 11 | **RM** | `rm` | Reward model training (Bradley-Terry) |

#### DPO and Direct Variants

| # | Algorithm | Key | Description |
|---|-----------|-----|-------------|
| 12 | **DPO** | `dpo` | Direct Preference Optimization |
| 13 | **IPO** | `ipo` | Identity Preference Optimization |
| 14 | **RDPO** | `rdpo` | Length-regularized DPO |
| 15 | **cDPO** | `cdpo` | Conservative DPO (label smoothing) |
| 16 | **ODPO** | `odpo` | DPO with offset |
| 17 | **DPOP** | `dpop` | DPO-Positive / Smaug |
| 18 | **DPO-Shift** | `dposhift` | Distribution-shifted DPO |
| 19 | **DPO+NLL** | `dpnll` | DPO with NLL regularization |
| 20 | **C2-DPO** | `c2dpo` | Constrained Controlled DPO |

#### Robust and Calibrated Variants

| # | Algorithm | Key | Description |
|---|-----------|-----|-------------|
| 21 | **BetaDPO** | `betadpo` | Dynamic beta per sample |
| 22 | **CalDPO** | `caldpo` | Calibrated DPO |
| 23 | **RobustDPO** | `robust_dpo` | Unbiased DPO under label noise |
| 24 | **DrDPO** | `drdpo` | Distributionally Robust DPO (ICLR 2025) |
| 25 | **AlphaDPO** | `alpha_dpo` | Adaptive reward margin (ICML 2025) |
| 26 | **MinorDPO** | `minor_dpo` | Minority-aware DPO |

#### Reference-Free Methods

| # | Algorithm | Key | Description |
|---|-----------|-----|-------------|
| 27 | **SimPO** | `simpo` | Simple Preference Optimization (no ref model) |
| 28 | **ORPO** | `orpo` | Odds Ratio Preference Optimization |
| 29 | **CPO** | `cpo` | Contrastive Preference Optimization |
| 30 | **CPO-SimPO** | `cposimpo` | CPO + SimPO combined |
| 31 | **AlphaPO** | `alphapo` | Reward shape transformation for SimPO |

#### Alternative Loss Functions

| # | Algorithm | Key | Description |
|---|-----------|-----|-------------|
| 32 | **Hinge** | `hinge` | Hinge loss preference optimization |
| 33 | **NCA** | `nca` | Noise Contrastive Alignment |
| 34 | **SPPO** | `sppo` | Self-Play Preference Optimization |
| 35 | **BCO** | `bco` | Binary Classifier Optimization |
| 36 | **EXO** | `exo` | Reverse KL preference optimization |
| 37 | **AOT** | `aot` | Alignment via Optimal Transport |
| 38 | **APO** | `apo` | Anchored Preference Optimization |

#### Generalized and Focal Methods

| # | Algorithm | Key | Description |
|---|-----------|-----|-------------|
| 39 | **GPO** | `gpo` | Generalized Preference Optimization (ICML 2024) |
| 40 | **FocalPO** | `focalpo` | Focal preference optimization |
| 41 | **WPO** | `wpo` | Weighted Preference Optimization (EMNLP 2024) |
| 42 | **fDPO** | `fdpo` | f-Divergence DPO (ICLR 2024) |
| 43 | **HDPO** | `hdpo` | Entropy Controllable DPO |
| 44 | **DiscoPOP** | `discopop` | Log-ratio modulated loss |
| 45 | **BPO** | `bpo` | Balanced Preference Optimization |
| 46 | **SamPO** | `sampo` | Sample-level Preference Optimization |
| 47 | **ChiPO** | `chipo` | Chi-squared Preference Optimization |
| 48 | **SPO** | `spo` | Spectral Preference Optimization |

#### Other Training Methods

| # | Algorithm | Key | Description |
|---|-----------|-----|-------------|
| 49 | **KTO** | `kto` | Kahneman-Tversky Optimization |
| 50 | **OnlineDPO** | `online_dpo` | Online DPO with iterative data |
| 51 | **SPIN** | `spin` | Self-Play Fine-Tuning |


---
## 5. Links

[![GitHub stars](https://img.shields.io/github/stars/warlockee/oxRL?style=social)](https://github.com/warlockee/oxRL)

- **GitHub:** [github.com/warlockee/oxRL](https://github.com/warlockee/oxRL)
- **PyPI:** [pypi.org/project/oxrl](https://pypi.org/project/oxrl/)
- **Docs:** See `registry/examples/` for ready-made configs for every algorithm